<!--
---
PURPOSE: Neural-behavior fusion, correlation analysis, and modeling.
REQUIRES:
  - outputs/neural/session_{id}_spike_times.npz
  - outputs/pose/session_{id}_pose_features.parquet
  - outputs/behavior/session_{id}_trials.parquet (optional)
  - outputs/eye/session_{id}_eye_features.parquet (optional)
PRODUCES:
  - outputs/fusion/session_{id}_fusion.parquet
  - outputs/models/session_{id}_alignment_report.json
  - outputs/models/session_{id}_model_{name}.joblib
  - outputs/models/session_{id}_metrics.json
WHAT_NEXT: notebooks/09_End_to_End_Run_and_QC_Checklist.ipynb
---
-->

# 08 Neural-Behavior Fusion and Modeling

**Core question: Do changes in behavior align with changes in neural activity?**

This notebook answers that question through six complementary analyses:

1. **Peri-Event Time Histograms (PETHs)** — How does neural firing change around behavioral events?
2. **Cross-Correlation** — What is the time lag between neural and behavioral changes?
3. **Sliding-Window Correlation** — When during the session is neural-behavior coupling strongest?
4. **Encoding Model** — Can behavior predict neural activity? (behavior → neural)
5. **Decoding Model** — Can neural activity predict behavior? (neural → behavior)
6. **Granger Causality** — Does neural activity *cause* behavior changes, or vice versa?

**Requires** (from earlier notebooks):
- Spike times (NB02)
- Pose features (NB07)
- Trial data (NB03, optional)
- Eye features (NB04, optional)

## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [18]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
# Works whether JupyterLab is launched from repo root or from notebooks/
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC  = ROOT / "src"

# put repo + src on sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


#print("ROOT:", ROOT)
#print("SRC :", SRC, "| exists:", SRC.exists())
#print("sys.path[0:3]:", sys.path[:3])

## Step 1: Load all modalities

We load neural, behavioral, and pose data for each session, then align everything to a common time grid.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

from config import get_config, make_provenance
from io_sessions import load_sessions_csv, get_session_bundle
from timebase import write_parquet_with_timebase, build_time_grid
from modeling import build_fusion_table
from neural_events import (
    compute_peth,
    compute_population_peth,
    trial_averaged_rates,
    build_population_vectors,
    reduce_population,
    screen_selective_units,
)
from cross_correlation import (
    crosscorrelation,
    population_crosscorrelation,
    sliding_correlation,
    fit_encoding_model,
    fit_decoding_model,
    fit_multi_covariate_encoding_model,
    granger_test,
    compute_neural_behavior_alignment,
)
from viz import (
    plot_peth,
    plot_population_peth,
    plot_trial_comparison,
    plot_crosscorrelation,
    plot_sliding_correlation,
    plot_encoding_decoding,
    plot_granger_summary,
    plot_unit_lag_distribution,
    plot_fusion_sanity,
)

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()

# --- Configuration ---
BIN_SIZE = cfg.bin_size_s          # time bin width (seconds)
PETH_WINDOW = (-0.5, 1.0)         # window around events (seconds)
PETH_BIN_SIZE = 0.01              # PETH bin size (seconds)
MAX_LAG_BINS = 40                  # max lag for cross-correlation
BEHAVIOR_COL  = "pose_speed"        # fallback single-column mode
BEHAVIOR_COLS = ["running", "pupil"] # multi-covariate mode (variance partitioning)
GAP_BINS      = 20                  # 500ms temporal gap in CV to prevent leakage
N_PERMUTATIONS = 200                # circular-shift permutations for null distribution

print(f"Sessions: {SESSION_IDS}")
print(f"Bin size: {BIN_SIZE}s | PETH window: {PETH_WINDOW} | Max lag: {MAX_LAG_BINS * BIN_SIZE:.2f}s")

## Step 2: Per-session neural-behavior analysis

For each session, we run the full analysis pipeline.

In [ ]:
all_reports = {}

for sid in SESSION_IDS:
    print(f"
{'='*70}")
    print(f"SESSION {sid}")
    print(f"{'='*70}")

    # --- Load data ---
    bundle = get_session_bundle(sid, sessions)
    units, spikes = bundle.load_spikes()

    if not spikes:
        print("  No spike data available. Skipping.")
        continue

    print(f"  Neural: {len(spikes)} units")

    # Load pose features
    features_path = cfg.outputs_dir / "pose" / f"session_{sid}_pose_features.parquet"
    pose_features = pd.read_parquet(features_path) if features_path.exists() else None
    if pose_features is not None:
        print(f"  Pose features: {pose_features.shape} | columns: {list(pose_features.columns[:6])}...")
    else:
        print("  Pose features: not available")

    # Load trials
    trials_path = cfg.outputs_dir / "behavior" / f"session_{sid}_trials.parquet"
    trials = pd.read_parquet(trials_path) if trials_path.exists() else None
    if trials is None:
        trials, _ = bundle.load_trials_and_events()
    if trials is not None:
        print(f"  Trials: {len(trials)} | go: {int(trials.get('go', pd.Series([0])).sum())} catch: {int(trials.get('catch', pd.Series([0])).sum())} hit: {int(trials.get('hit', pd.Series([0])).sum())} miss: {int(trials.get('miss', pd.Series([0])).sum())}")

    # Load running speed (NWB wheel encoder)
    running_path = cfg.outputs_dir / "behavior" / f"session_{sid}_running.parquet"
    running_df = pd.read_parquet(running_path) if running_path.exists() else bundle.load_running_speed()
    if running_df is not None:
        print(f"  Running: {len(running_df)} samples, mean={running_df['running'].mean():.1f} cm/s")
    else:
        print("  Running speed: not available")

    # Load eye features
    eye_path = cfg.outputs_dir / "eye" / f"session_{sid}_eye_features.parquet"
    eye_features = pd.read_parquet(eye_path) if eye_path.exists() else None
    if eye_features is not None:
        print(f"  Eye features: {eye_features.shape}")

    # --- Build fusion table ---
    motifs_path = cfg.outputs_dir / "pose" / f"session_{sid}_motifs.parquet"
    motifs = pd.read_parquet(motifs_path) if motifs_path.exists() else None

    fusion = build_fusion_table(spikes, motifs, BIN_SIZE)
    fusion_path = cfg.outputs_dir / "fusion" / f"session_{sid}_fusion.parquet"
    write_parquet_with_timebase(
        fusion, fusion_path,
        provenance=make_provenance(sid, "nwb"),
        required_columns=["t"],
    )
    print(f"  Fusion table: {fusion.shape} ({fusion['t'].min():.2f}s to {fusion['t'].max():.2f}s)")

    # ============================================================
    # ANALYSIS 1: Peri-Event Time Histograms
    # ============================================================
    print(f"
  --- PETHs ---")
    if trials is not None and "t" in trials.columns:
        event_times = trials["t"].dropna().to_numpy()
        print(f"  Computing PETHs for {len(event_times)} trial events...")

        pop_peth = compute_population_peth(spikes, event_times, PETH_WINDOW, PETH_BIN_SIZE)
        print(f"  Population PETH: {pop_peth['population_matrix'].shape}")
        plot_population_peth(pop_peth, title=f"Session {sid} - Population PETH (all trials)")

        # Trial-averaged by go/catch (change detection task)
        for group_col in ["go", "hit", "trial_type"]:
            if group_col in trials.columns:
                tavg = trial_averaged_rates(spikes, trials, group_col, PETH_WINDOW, PETH_BIN_SIZE)
                if tavg:
                    print(f"  Trial-averaged by {group_col}: {list(tavg.keys())}")
                    plot_trial_comparison(tavg)
                break
    else:
        print("  No trial data available for PETHs.")

    # ============================================================
    # ANALYSIS 2-6: Cross-correlation and encoding models
    # ============================================================
    # Merge all behavioral signals into one DataFrame aligned on 't'.
    beh_parts = []
    if running_df is not None and "t" in running_df.columns:
        beh_parts.append(running_df[["t", "running"]])
    if pose_features is not None and "t" in pose_features.columns:
        beh_parts.append(pose_features)
    if eye_features is not None and "t" in eye_features.columns:
        ef = eye_features.copy()
        if "pupil" not in ef.columns:
            pupil_candidates = [c for c in ef.columns if "pupil" in c.lower() and c != "t"]
            if pupil_candidates:
                ef = ef.rename(columns={pupil_candidates[0]: "pupil"})
        beh_parts.append(ef)

    if beh_parts:
        from functools import reduce
        behavior_df = reduce(
            lambda a, b: pd.merge_asof(
                a.sort_values("t"), b.sort_values("t"),
                on="t", direction="nearest", tolerance=BIN_SIZE * 2,
            ),
            beh_parts,
        )
    else:
        behavior_df = None

    avail_cols = [c for c in BEHAVIOR_COLS if behavior_df is not None and c in behavior_df.columns]
    use_multi = len(avail_cols) >= 2

    if behavior_df is not None and (use_multi or BEHAVIOR_COL in (behavior_df.columns if behavior_df is not None else [])):
        print(f"
  --- Neural-behavior alignment (covariates: {avail_cols if use_multi else [BEHAVIOR_COL]}) ---")

        report = compute_neural_behavior_alignment(
            spike_times_dict=spikes,
            behavior_df=behavior_df,
            trials=trials,
            bin_size=BIN_SIZE,
            behavior_col=BEHAVIOR_COL,
            behavior_cols=avail_cols if use_multi else None,
            max_lag_bins=MAX_LAG_BINS,
            gap_bins=GAP_BINS,
            n_permutations=N_PERMUTATIONS,
            run_variance_partitioning=use_multi,
        )
        all_reports[sid] = report

        enc = report.get("encoding", {})
        print(f"  Units: {report.get('n_units', 0)} | Time bins: {report.get('n_timebins', 0)}")
        print(f"  Peak cross-corr: r={report.get('peak_corr', 0):.4f} at lag={report.get('peak_lag_s', 0):.3f}s")
        if "full_r2" in enc:
            print(f"  Encoding full R2 (all covariates): {enc['full_r2']:.4f}")
            for cov, ur2 in enc.get("unique_r2", {}).items():
                print(f"    Unique R2 [{cov}]: {ur2:.4f}")
            pval = enc.get("perm_p_value")
            if pval is not None:
                print(f"  Permutation p={pval:.3f}  z={enc.get('perm_z_score', 0):.1f}")
        else:
            print(f"  Encoding R2 (behavior->neural): {enc.get('mean_r2', float('nan')):.4f}")
            pval = enc.get("perm_p_value")
            if pval is not None:
                print(f"  Permutation p={pval:.3f}  z={enc.get('perm_z_score', 0):.1f}")
        print(f"  Decoding R2 (neural->behavior): {report.get('decoding', {}).get('mean_r2', float('nan')):.4f}")

        gc_n2b = report.get("granger_neural_to_behavior", {})
        gc_b2n = report.get("granger_behavior_to_neural", {})
        print(f"  Granger neural->behavior: F={gc_n2b.get('f_statistic', 0):.2f}, p={gc_n2b.get('p_value', 1):.4f}")
        print(f"  Granger behavior->neural: F={gc_b2n.get('f_statistic', 0):.2f}, p={gc_b2n.get('p_value', 1):.4f}")

        report_path = cfg.outputs_dir / "models" / f"session_{sid}_alignment_report.json"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        def _serialize(obj):
            if isinstance(obj, np.ndarray): return obj.tolist()
            if isinstance(obj, np.floating): return float(obj)
            if isinstance(obj, np.integer): return int(obj)
            if isinstance(obj, pd.DataFrame): return obj.to_dict()
            return obj
        import json as _json
        serializable = _json.loads(_json.dumps(report, default=_serialize))
        report_path.write_text(_json.dumps(serializable, indent=2, default=str))
        print(f"  Report saved: {report_path}")
    else:
        print(f"
  Skipping alignment analysis (no behavioral data available)")

print(f"
{'='*70}")
print(f"Processed {len(all_reports)} sessions with full alignment analysis.")


## Step 3: Visualize results

Detailed plots for each analysis. These help you understand *how* neural activity and behavior are coupled.

In [ ]:
import matplotlib.pyplot as plt

for sid, report in all_reports.items():
    print(f"\n{'='*70}")
    print(f"VISUALIZATIONS - Session {sid}")
    print(f"{'='*70}")

    # --- Cross-correlation ---
    xcorr = report.get("crosscorrelation")
    if xcorr:
        plot_crosscorrelation(xcorr, bin_size=BIN_SIZE)
        plt.show()

    # --- Per-unit lag distribution ---
    unit_xcorr = report.get("unit_crosscorrelation")
    if unit_xcorr is not None and not unit_xcorr.empty:
        plot_unit_lag_distribution(unit_xcorr, bin_size=BIN_SIZE)
        plt.show()

    # --- Sliding-window correlation ---
    slide = report.get("sliding_correlation")
    if slide:
        plot_sliding_correlation(slide, bin_size=BIN_SIZE)
        plt.show()

    # --- Encoding vs Decoding ---
    enc = report.get("encoding", {})
    dec = report.get("decoding", {})
    if enc.get("cv_scores") or dec.get("cv_scores"):
        plot_encoding_decoding(enc, dec)
        plt.show()

    # --- Granger causality ---
    gc_n2b = report.get("granger_neural_to_behavior", {})
    gc_b2n = report.get("granger_behavior_to_neural", {})
    if gc_n2b or gc_b2n:
        plot_granger_summary(gc_n2b, gc_b2n)
        plt.show()

## Step 4: Unit selectivity screening

Which neurons are most responsive to specific behavioral conditions?

In [ ]:
for sid in SESSION_IDS:
    bundle = get_session_bundle(sid, sessions)
    units, spikes = bundle.load_spikes()
    if not spikes:
        continue

    trials_path = cfg.outputs_dir / "behavior" / f"session_{sid}_trials.parquet"
    trials = pd.read_parquet(trials_path) if trials_path.exists() else None
    if trials is None or "t" not in trials.columns:
        print(f"[session {sid}] No trial data for selectivity screening")
        continue

    # Prefer go/catch (change-detection task columns), fall back to trial_type
    cond_a, cond_b, label_a, label_b = None, None, None, None
    if "go" in trials.columns and "catch" in trials.columns:
        mask_a = trials["go"].astype(bool)
        mask_b = trials["catch"].astype(bool)
        cond_a = trials[mask_a & trials["t"].notna()]
        cond_b = trials[mask_b & trials["t"].notna()]
        label_a, label_b = "go", "catch"
    elif "hit" in trials.columns and "miss" in trials.columns:
        mask_a = trials["hit"].astype(bool)
        mask_b = trials["miss"].astype(bool)
        cond_a = trials[mask_a & trials["t"].notna()]
        cond_b = trials[mask_b & trials["t"].notna()]
        label_a, label_b = "hit", "miss"
    elif "trial_type" in trials.columns:
        types = trials["trial_type"].dropna().unique()
        if len(types) >= 2:
            cond_a = trials[trials["trial_type"] == types[0]]
            cond_b = trials[trials["trial_type"] == types[1]]
            label_a, label_b = str(types[0]), str(types[1])

    if cond_a is None or len(cond_a) < 5 or len(cond_b) < 5:
        print(f"[session {sid}] Not enough trials for selectivity (need >=5 per condition)")
        continue

    times_a = cond_a["t"].dropna().to_numpy()
    times_b = cond_b["t"].dropna().to_numpy()
    print(f"
[session {sid}] Selectivity: '{label_a}' ({len(times_a)}) vs '{label_b}' ({len(times_b)})")

    selectivity = screen_selective_units(spikes, times_a, times_b, window=(0.0, 0.5))
    if selectivity.empty:
        print("  No units to screen.")
        continue

    n_sig = selectivity["significant"].sum()
    print(f"  {n_sig}/{len(selectivity)} units significantly selective (p < 0.05)")
    print(selectivity.head(10)[["unit_id", "d_prime", "rate_diff", "p_value", "significant"]].to_string(index=False))

    sel_path = cfg.outputs_dir / "models" / f"session_{sid}_selectivity.parquet"
    sel_path.parent.mkdir(parents=True, exist_ok=True)
    write_parquet_with_timebase(
        selectivity, sel_path,
        provenance=make_provenance(sid, "selectivity_screen"),
    )

    if n_sig > 0:
        best_unit = str(selectivity.iloc[0]["unit_id"])
        if best_unit in spikes:
            peth_a = compute_peth(spikes[best_unit], times_a, PETH_WINDOW, PETH_BIN_SIZE)
            peth_b = compute_peth(spikes[best_unit], times_b, PETH_WINDOW, PETH_BIN_SIZE)

            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(7, 3))
            ax.plot(peth_a["time_bins"], peth_a["mean_rate"], color="#2563eb", label=f"{label_a} (n={peth_a['n_trials']})")
            ax.fill_between(peth_a["time_bins"], peth_a["mean_rate"] - peth_a["sem_rate"], peth_a["mean_rate"] + peth_a["sem_rate"], alpha=0.15, color="#2563eb")
            ax.plot(peth_b["time_bins"], peth_b["mean_rate"], color="#ef4444", label=f"{label_b} (n={peth_b['n_trials']})")
            ax.fill_between(peth_b["time_bins"], peth_b["mean_rate"] - peth_b["sem_rate"], peth_b["mean_rate"] + peth_b["sem_rate"], alpha=0.15, color="#ef4444")
            ax.axvline(0, color="gray", linestyle="--", linewidth=1)
            d = selectivity.iloc[0]["d_prime"]
            ax.set_title(f"Most selective unit ({best_unit}): d'={d:.2f}")
            ax.set_xlabel("Time from event (s)")
            ax.set_ylabel("Firing rate (Hz)")
            ax.legend(fontsize=8, frameon=False)
            ax.grid(True, alpha=0.15)
            plt.tight_layout()
            plt.show()


In [ ]:
print("
" + "=" * 70)
print("NEURAL-BEHAVIOR ALIGNMENT SUMMARY")
print("=" * 70)

for sid, report in all_reports.items():
    enc = report.get("encoding", {})
    dec = report.get("decoding", {})
    gc_n2b = report.get("granger_neural_to_behavior", {})
    gc_b2n = report.get("granger_behavior_to_neural", {})

    print(f"
Session {sid}:")
    print(f"  Units: {report.get('n_units', 'N/A')} | Bins: {report.get('n_timebins', 'N/A')}")
    print(f"  Cross-corr peak: r={report.get('peak_corr', 0):.4f} at {report.get('peak_lag_s', 0):.3f}s")

    if "full_r2" in enc:
        print(f"  Encoding full R2 (multi-covariate): {enc['full_r2']:.4f}")
        for cov, ur2 in enc.get("unique_r2", {}).items():
            print(f"    Unique R2 [{cov}]: {ur2:.4f}")
    else:
        enc_r2 = enc.get("mean_r2", float("nan"))
        print(f"  Encoding R2 (behavior->neural): {enc_r2:.4f}")

    dec_r2 = dec.get("mean_r2", float("nan"))
    print(f"  Decoding R2 (neural->behavior): {dec_r2:.4f}")

    n2b_sig = "YES" if gc_n2b.get("p_value", 1) < 0.05 else "no"
    b2n_sig = "YES" if gc_b2n.get("p_value", 1) < 0.05 else "no"
    print(f"  Granger causal? neural->behavior: {n2b_sig} | behavior->neural: {b2n_sig}")

if not all_reports:
    print("
  No sessions had both neural and behavioral data.")
    print("  Run Notebooks 02 and 03 first, then re-run this notebook.")

print("
Notebook 08 complete. Run Notebook 09 for QC checklist.")
